# §0 Setup & Configuration

This section establishes the **reproducible foundation** for the entire notebook.
Every subsequent phase (§2 catchment, §3 demographics, §4 competitors, §5 demand,
§6 financial model, §7 scenarios, §8 report) depends on the paths, keys, and
parameters defined here.

**v1 flaws addressed in this section:**
- *Hardcoded Google Drive paths* (v1 used literal Drive mount-point paths
  that only worked on the original author's Colab) — replaced with
  `PROJECT_ROOT`-relative `pathlib.Path`s that work on any machine.
- *Drive mount dependency* (v1 called the Drive mount API) — replaced with
  `git clone`, so caches travel with the repo and no manual Drive setup
  is needed.
- *Scattered constants* — consolidated into a single `BASE_ASSUMPTIONS` dict.

The **dual-environment pattern** detects Colab vs Windows at runtime and resolves
all paths accordingly. This is what makes "Restart & Run All" work in both
environments without manual intervention (PIPE-01, PIPE-02).

In [ ]:
# §0.1 — Installs & imports
# Only pip-install packages NOT preinstalled in Colab (RESEARCH.md §3).
# geopandas/shapely/pyproj/folium/pandas/matplotlib/jinja2 are preinstalled.
import os, sys, json, requests
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # requests-cache and python-dotenv are NOT preinstalled in Colab
    %pip install -q requests-cache python-dotenv

from requests_cache import CachedSession

print("[setup] imports loaded")

## §0.2 Environment Detection & Paths

v1 used hardcoded Google Drive mount-point paths that only worked on the
original author's Colab with a manually mounted Drive. This cell detects the
runtime environment and resolves `PROJECT_ROOT` via `pathlib` so the same code
works on Colab (after `git clone`) and on Windows.

**Key design decisions (from CONTEXT.md):**
- **D-06:** Colab clones the repo into `/content/medical-clinic`; caches come
  with the clone (committed to git per D-12).
- **D-07:** No Drive mount — explicitly rejected. Caches and code travel
  together via git.
- **D-12:** API response caches are committed to git, so a fresh clone can
  re-run the notebook offline/keyless at $0 (PIPE-04).

The API key loads from Colab Secrets (`userdata`) or local `.env`
(`python-dotenv`). A missing key degrades gracefully — cache hits satisfy
all fetches (PIPE-03).

In [ ]:
# §0.2 — Environment detection & path resolution (PIPE-02)
# TODO: replace <OWNER> with your GitHub username
if IN_COLAB:
    # Clone repo if not already present (caches come with it — D-06)
    import subprocess
    repo_path = "/content/medical-clinic"
    if not os.path.exists(repo_path):
        subprocess.run(["git", "clone", "https://github.com/jatwell93/medical-clinic.git", repo_path],
                       check=True)
    PROJECT_ROOT = Path(repo_path)
    # Load API key from Colab Secrets
    try:
        from google.colab import userdata
        GOOGLE_PLACES_KEY = userdata.get("GOOGLE_PLACES_KEY")
    except Exception:
        GOOGLE_PLACES_KEY = ""
else:
    # Local Windows — load from .env via python-dotenv
    from dotenv import load_dotenv
    load_dotenv()  # Fails silently if .env missing — acceptable (cache fallback)
    GOOGLE_PLACES_KEY = os.environ.get("GOOGLE_PLACES_KEY", "")
    PROJECT_ROOT = Path.cwd()  # Notebooks don't have __file__

# Graceful degradation: key is optional if cache is populated (D-12)
CACHE_DIR = PROJECT_ROOT / "data" / "cache"
assert CACHE_DIR.exists() or GOOGLE_PLACES_KEY, \
    "Need GOOGLE_PLACES_KEY or a populated data/cache/ to run."

print(f"[env] IN_COLAB={IN_COLAB}, PROJECT_ROOT={PROJECT_ROOT}")
print(f"[env] Google Places key: {'loaded' if GOOGLE_PLACES_KEY else 'empty (cache-only mode)'}")

## §0.3 Parameters & Assumptions

This is the **single source of truth** for every numeric assumption in the
notebook (PIPE-05). v1 had constants scattered across cells — making it
impossible to audit or run scenarios without hunting through the notebook.

**Why a flat dict?** The flat-dict shape makes Phase 5 scenarios trivial:
`{**BASE_ASSUMPTIONS, **overrides}`. A `@dataclass` would break this pattern;
an external YAML/JSON file would break the "single parameters cell" wording.

**Rule (from ARCHITECTURE.md):** any numeric literal in §2–§8 that isn't a
unit conversion is a bug. Downstream cells *read* these values; they never
redefine them.

Every value has an inline comment with a **source citation** or an
`# UNCONFIRMED` flag so the assumptions register (Phase 5, REP-01) can be
assembled automatically from this cell.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  SINGLE PARAMETERS CELL — every assumption lives here (PIPE-05)
#  Downstream cells READ these; they never redefine them.
#  Scenarios (Phase 5): {**BASE_ASSUMPTIONS, **overrides}
# ═══════════════════════════════════════════════════════════════

SITE_ADDRESS = "292-296 Johnston St, Abbotsford VIC 3067"
CATCHMENT_RADII_M = [1000, 3000, 5000]  # 1/3/5 km rings
PEER_POSTCODES = ["3067", "3066", "3068", "3070", "3078", "3079",
                  "3101", "3121", "3122", "3123"]

# Cache control — single global refresh flag (D-04)
FORCE_REFRESH = False

BASE_ASSUMPTIONS = {
    # --- Site & catchment ---
    "site_address":           SITE_ADDRESS,
    "catchment_radii_m":      CATCHMENT_RADII_M,
    "peer_postcodes":         PEER_POSTCODES,

    # --- Demand (source: MBS SA3 20604 Yarra; RACGP GP:pop benchmarks) ---
    "consults_per_capita_yr": {"0-14": 4.4, "15-64": 5.1, "65+": 11.0},  # MBS 2024-25 — UNCONFIRMED vintage
    "consults_per_gp_day":    28,          # RACGP typical full-book GP

    # --- Revenue (source: MBS item 23 rebate; local gap-fee scan) ---
    "bulk_bill_share":        0.70,        # DECISION: mixed-billing model (70% bulk / 30% private)
    "std_rebate":             43.90,       # MBS item 23, 2025-26 — verify at build time
    "private_fee":            95.00,       # inner-Melb typical standard consult — UNCONFIRMED
    "gp_revenue_share":       0.35,        # % of billings retained by clinic (GPs keep 65%) — UNCONFIRMED range 30-35%

    # --- Costs ---
    "rent_yr":                100_000,     # UNCONFIRMED — key sensitivity, flagged in report
    "fitout_capital":         350_000,     # estimate — flagged, sensitivity-tested
    "n_gp_fte":               5,
    "n_allied_fte":           1,

    # --- Operational ---
    "days_per_yr":            220,         # GP working days after leave
    "utilisation":            0.75,        # ramp-up target utilisation

    # --- Phase 2: Catchment (source: ABS ASGS Edition 3; PITFALLS.md Pitfall 1/2) ---
    "sa1_shapefile":          "SA1_2021_AUST_GDA2020.zip",  # ABS ASGS Edition 3 — manual download
    "poa_shapefile":          "POA_2021_AUST_GDA2020_SHP.zip",  # ABS ASGS Edition 3 — on hand
    "assert_3km_buffer_km2":  28.27,       # π×3² — sanity check for EPSG:7855 buffering (GEO-02, D-09a)
    "assert_3km_buffer_tol":  0.1,         # km² tolerance
    "catchment_pop_plausible_range": (80_000, 110_000),  # inner-Melbourne 3km ring (D-09b, PITFALLS.md Pitfall 2)
    "erp_vintage_year":       2024,        # ABS Regional Population 2023-24 FY (released March 2025) — DEMO-04
}

print(f"[params] {len(BASE_ASSUMPTIONS)} assumptions loaded")
print(f"[params] FORCE_REFRESH={FORCE_REFRESH}")

# §1 Data Acquisition & Caching

v1 had **no caching** — every re-run hit the ABS and Google APIs, costing
money and making the notebook non-reproducible offline. This section
establishes a single `CachedSession` that **all** external calls go through
(ABS Data API, Google Geocoding, Google Places).

**What this gives us (PIPE-04):**
- Re-runs are **free** — cached responses are served from disk.
- Re-runs are **offline-capable** — no network needed if cache is populated.
- Re-runs are **keyless** — ABS needs no key; Places/Geocode cache hits
  need no key. A grader or investor can clone the repo and run the entire
  notebook without any API keys.

The ABS smoke test below proves the cache works using a **keyless** endpoint
(the dataflow list requires no API key). On first run it fetches from the
network; on second run it serves from cache (`response.from_cache == True`).

In [ ]:
# §1.1 — Cache session construction (D-01, D-02, D-03)
# Single cached HTTP boundary for ALL external calls (D-01, D-02, D-03)
# requests-cache filesystem backend: one JSON file per response in data/cache/
# allowable_methods includes POST for Google Places API (New) — STACK.md requirement
CACHE_DIR.mkdir(parents=True, exist_ok=True)

session = CachedSession(
    cache_name=str(CACHE_DIR / "api_cache"),
    backend='filesystem',
    allowable_methods=('GET', 'POST'),  # POST needed for Places API (New)
    expire_after=-1,                     # No expiry — census is immutable vintage
)

# Apply FORCE_REFRESH flag from PARAMS cell (D-04)
if FORCE_REFRESH:
    session.cache.clear()
    print("[cache] cleared (FORCE_REFRESH=True)")

print(f"[cache] session ready → {CACHE_DIR}")

## §1.2 ABS Data API Smoke Test

This proves the cache layer works using a **keyless** ABS endpoint. The
dataflow list (`/rest/dataflow?detail=allstubs`) requires no API key, so
this test runs even in cache-only mode (no `GOOGLE_PLACES_KEY`).

**Important format note:** the ABS REST API returns **SDMX-ML XML** by
default (Content-Type: `application/vnd.sdmx.structure+xml`), not JSON.
This is the standard SDMX format for statistical data exchange. The actual
census data fetch in Phase 2 will use `?format=csvfilewithlabels` to get
human-readable CSV with both dimension codes and labels — but this smoke
test only validates that the cache layer works, so we accept the XML
structure response and verify it looks like SDMX.

**What it validates (PIPE-04):**
- The `CachedSession` is correctly constructed and can make HTTP requests.
- On first run: fetches from the network and writes a cache file.
- On second run: serves from cache (`response.from_cache == True`).
- Cache files appear in `data/cache/`.

This is the foundation that makes every subsequent API call (ABS census,
Google geocode, Google Places) free and offline-reproducible.

In [ ]:
# §1.2 — ABS Data API smoke test (PIPE-04 validation)
# Keyless smoke test — proves the cache layer works (PIPE-04)
# ABS dataflow list endpoint requires no API key.
# NOTE: ABS REST API returns SDMX-ML XML by default (Content-Type:
# application/vnd.sdmx.structure+xml), NOT JSON. The data pipeline (Phase 2)
# will use ?format=csvfilewithlabels for actual census data; this smoke test
# only validates the cache layer, so we accept the XML structure response.
ABS_DATAFLOW_URL = "https://data.api.abs.gov.au/rest/dataflow?detail=allstubs"

response = session.get(ABS_DATAFLOW_URL)
print(f"[abs] HTTP {response.status_code}, from_cache={response.from_cache}")
print(f"[abs] Content-Type: {response.headers.get('Content-Type', 'unknown')}")

# Validate response — ABS returns SDMX-ML XML, not JSON
assert response.status_code == 200, f"ABS returned HTTP {response.status_code}"
content_type = response.headers.get("Content-Type", "")
assert "sdmx" in content_type or "xml" in content_type, \
    f"Unexpected Content-Type: {content_type}"
assert len(response.content) > 0, "Empty response body"
# Sanity check: SDMX structure messages contain the sdmx namespace
assert b"sdmx" in response.content[:2000], \
    "Response body does not look like SDMX-ML"

# Cache validation assertions (RESEARCH.md Validation Architecture)
# requests-cache filesystem backend stores responses in a subdirectory
# named after cache_name (e.g. data/cache/api_cache/*.json), so use rglob.
assert CACHE_DIR.exists(), "Cache directory not created"
cache_files = list(CACHE_DIR.rglob("*.json"))
assert len(cache_files) > 0, "No cache files created — cache layer not working"

print(f"[abs] {len(cache_files)} cache file(s) in {CACHE_DIR}")
print(f"[abs] Smoke test PASSED — cache layer operational")

## §1.2 Geocode Site

v1 used the **postcode centroid** as the catchment centre (PITFALLS.md Pitfall 3)
and referenced the site coordinates before they were defined (out-of-order
execution). This cell geocodes the **exact address** ONCE through the cached
session, prints the coordinates, and asks the reader to visually verify the pin
on the immediately-following map before proceeding.

The cached response means re-runs are **free and keyless** — after the first
successful geocode, the coordinates are served from `data/cache/` and no
`GOOGLE_PLACES_KEY` is needed (PIPE-04).

**Known anchor (D-31):** The site is at the Johnston St × Hoddle St corner,
Abbotsford. Expected coordinates ≈ (-37.799, 145.003). The map below lets you
confirm the pin lands on the right spot before any buffer or Places search
centres on it.

In [ ]:
# Geocode the exact site address through the Phase 1 CachedSession (D-31, PITFALLS.md Pitfall 3)
# v1 flaw: used postcode centroid + referenced site coords before definition (out-of-order execution)
import urllib.parse

GEOCODE_URL = (
    "https://maps.googleapis.com/maps/api/geocode/json?address="
    + urllib.parse.quote(SITE_ADDRESS)
    + f"&key={GOOGLE_PLACES_KEY}"
)

# Cached fetch — re-runs are free and keyless after first run (PIPE-04)
if GOOGLE_PLACES_KEY or any(CACHE_DIR.glob("geocode_*.json")):
    resp = session.get(GEOCODE_URL)
    data = resp.json()
    assert data.get("status") == "OK", f"Geocode failed: {data.get('status')} {data.get('error_message','')}"
    result = data["results"][0]
    site_lat = result["geometry"]["location"]["lat"]
    site_lon = result["geometry"]["location"]["lng"]
    site_place_id = result.get("place_id", "")
    print(f"[geocode] {SITE_ADDRESS}")
    print(f"[geocode] lat={site_lat:.6f}, lon={site_lon:.6f} (from_cache={resp.from_cache})")
    print(f"[geocode] ⚠ Visually verify this pin on the map below before proceeding.")
    print(f"[geocode]    Expected: Johnston St × Hoddle St corner, Abbotsford")
else:
    raise RuntimeError(
        "No GOOGLE_PLACES_KEY and no cached geocode response. "
        "Add the key via Colab Secrets or local .env, run once, then re-run keyless."
    )

In [ ]:
# §1.2 verification map — visually confirm the pin (D-31)
import folium
m_verify = folium.Map(location=[site_lat, site_lon], zoom_start=16, tiles="CartoDB Positron")
folium.Marker(
    [site_lat, site_lon],
    popup=f"Site: {SITE_ADDRESS}<br>({site_lat:.5f}, {site_lon:.5f})",
    tooltip="292-296 Johnston St",
    icon=folium.Icon(color="red", icon="info-sign"),
).add_to(m_verify)
m_verify

# §2 Geospatial Catchment

v1's headline flaw was **whole-postcode summing** (PITFALLS.md Pitfall 2):
it summed the FULL population of every postcode touching the buffer, inflating
catchment population 2–4× and flipping the go/no-go answer. This section fixes
it with **SA1-level area apportionment** (D-01).

All metric operations (buffer, area, distance) happen in **EPSG:7855**
(GDA2020 / MGA zone 55) — PITFALLS.md Pitfall 1 warns that buffering in
degrees creates a "buffer" spanning the continent. The CRS convention is:
**metric ops in EPSG:7855, display in EPSG:4326**.

The v1-vs-v2 comparison at the end (§2.4) is the teaching moment showing
v1 inflated 2–4×.

In [ ]:
# Load SA1 + POA boundary geometry from data/local/ (D-02, D-07)
# Both are gitignored — large, immutable, not committed
# Geospatial imports (preinstalled in Colab — STACK.md)
import geopandas as gpd
from shapely.geometry import Point

SA1_ZIP = PROJECT_ROOT / "data" / "local" / BASE_ASSUMPTIONS["sa1_shapefile"]
POA_ZIP = PROJECT_ROOT / "data" / "local" / BASE_ASSUMPTIONS["poa_shapefile"]

if not SA1_ZIP.exists():
    print(f"⚠ SA1 shapefile missing: {SA1_ZIP}")
    print(f"  Download from ABS ASGS Edition 3 digital boundary files (see .env.example)")
    print(f"  Falling back to POA-level apportionment (degraded — uniform-density assumption coarser)")
    USE_SA1 = False
else:
    USE_SA1 = True

poa = gpd.read_file(POA_ZIP)
poa = poa[poa["STE_NAME21"] == "Victoria"].to_crs("EPSG:7855")  # metric CRS for area math
print(f"[geom] POA: {len(poa)} VIC polygons, CRS={poa.crs.to_epsg()}")

if USE_SA1:
    sa1 = gpd.read_file(SA1_ZIP)
    sa1 = sa1[sa1["STE_NAME21"] == "Victoria"].to_crs("EPSG:7855")
    print(f"[geom] SA1: {len(sa1)} VIC polygons, CRS={sa1.crs.to_epsg()}")

## §2.2 Build 1/3/5 km Buffers (EPSG:7855)

The CRS convention is **metric ops in EPSG:7855, display in EPSG:4326**
(PITFALLS.md Pitfall 1). The 3 km buffer area assertion (≈ 28.27 km² = π×3²)
is the sanity check that catches a degree-based buffer before it poisons
everything downstream.

v1 buffered in degrees → the "buffer" spanned half of Victoria. The assertion
below would have caught it instantly.

In [ ]:
# Build 1/3/5 km buffers in EPSG:7855 (GEO-02, D-09a, PITFALLS.md Pitfall 1)
# v1 flaw: buffered in degrees → "buffer" spanning half of Victoria
site_point = gpd.GeoDataFrame(
    [{"name": "site"}], geometry=[Point(site_lon, site_lat)], crs="EPSG:4326"
)
site_metric = site_point.to_crs("EPSG:7855")

buffers = {}
for r in BASE_ASSUMPTIONS["catchment_radii_m"]:
    buf = site_metric.buffer(r)
    buffers[r] = buf
    area_km2 = buf.area.iloc[0] / 1e6
    print(f"[buffer] {r//1000} km: area={area_km2:.2f} km²")

# Sanity assertion (GEO-02, D-09a) — 3km buffer ≈ 28.27 km² (π×3²)
area_3k = buffers[3000].area.iloc[0] / 1e6
assert abs(area_3k - BASE_ASSUMPTIONS["assert_3km_buffer_km2"]) < BASE_ASSUMPTIONS["assert_3km_buffer_tol"], \
    f"3km buffer area {area_3k:.2f} km² ≠ {BASE_ASSUMPTIONS['assert_3km_buffer_km2']} km² — CRS bug (PITFALLS.md Pitfall 1)"
print(f"[buffer] ✓ 3km buffer area assertion passed ({area_3k:.2f} km² ≈ π×3²)")

## §2.3 SA1 Area Apportionment

This is the **headline v1 fix** (PITFALLS.md Pitfall 2). v1 summed the FULL
population of every postcode touching the buffer. v2 area-apportions each
SA1's population by the fraction of its area inside the buffer.

The **uniform-density assumption** is documented (D-05) and the Yarra River
is annotated on every map so readers see the geographic issue — the river and
industrial land in Abbotsford mean the assumption likely *overstates*
population in that slice.

**Note:** The SA1 total persons come from §3 (Plan 02-02) via
`fetch_sa1_total_persons(sa1_codes)`. This cell defines the apportionment
logic and spatial filter; the population weights are wired in §3 once the ABS
SA1 census is fetched.

In [ ]:
# SA1-level area apportionment (D-01 — the headline v1 fix, PITFALLS.md Pitfall 2)
# v1 flaw: summed FULL population of every postcode touching the buffer (2-4× overstatement)
# v2: area-weight each SA1's population by the fraction inside the buffer
if USE_SA1:
    # Spatial-filter SA1s intersecting the 5km buffer (D-08 — beat 30s API timeout)
    sa1_in_5k = sa1[sa1.geometry.intersects(buffers[5000].iloc[0])].copy()
    print(f"[apportion] {len(sa1_in_5k)} SA1s intersect 5km buffer")

    # Per-ring area-weighted apportionment
    # sa1_pop_df is produced in §3 (Plan 02-02) — stub here, wired there
    def apportion_ring(sa1_gdf, sa1_pop_df, ring_geom):
        """Area-weight SA1 populations into a ring. Returns total population in ring."""
        ring = gpd.GeoDataFrame(geometry=[ring_geom], crs="EPSG:7855")
        inter = sa1_gdf.overlay(ring, how="intersection")
        # Join population
        inter = inter.merge(sa1_pop_df, on="SA1_CODE21", how="left")
        # Area fraction (intersect_area / full_sa1_area)
        full_area = sa1_gdf.set_index("SA1_CODE21")["geometry"].area
        inter["frac"] = inter.geometry.area / inter["SA1_CODE21"].map(full_area).values
        inter["pop_in_ring"] = inter["frac"] * inter["Total_P_P"]
        return inter["pop_in_ring"].sum(), inter

    # NOTE: sa1_pop_df is assigned in §3 demographics (Plan 02-02).
    # For Phase 2 §2 standalone validation, we use a proxy: SA1 area-only weights
    # (uniform density across all SA1s in 5km). The real population weights land in §3.
    # This stub is overwritten by §3 once ABS SA1 census is fetched.
    print("[apportion] ℹ Population weights come from §3 (ABS C21_G01_SA1). "
          "Catchment population totals are computed in §3 after the SA1 census fetch.")
else:
    print("[apportion] ⚠ SA1 shapefile missing — using POA-level apportionment (degraded)")
    # POA-level fallback (coarser) — same area-weight logic at POA granularity

## §2.4 v1-vs-v2 Catchment Comparison

This is the **teaching moment** (D-06, success criterion #2). v1's naive
whole-postcode sum is reproduced inline so readers see the flawed logic next
to the fixed logic. The grouped bar chart + side-by-side table show v1
inflating 2–4× per ring.

The v1 and v2 totals come from §3 (Plan 02-02). This cell defines the comparison
function `compare_v1_v2(v1_ring_pops, v2_ring_pops)` that §3.3 calls after
computing both the v1 naive whole-postcode sums and the v2 apportioned totals.

In [ ]:
# v1-vs-v2 catchment comparison (D-06, success criterion #2)
# Reproduce v1's naive whole-postcode sum inline, compare against v2 apportioned
def v1_naive_catchment_pop(ring_geom_m, poa_pop_df):
    """v1 flaw: sum FULL POA population for any POA touching the buffer.
    poa_pop_df: DataFrame with columns POA_CODE21, Total_P_P (from §3 G01 fetch).
    Returns the summed population (int), NOT a GeoDataFrame."""
    touching = poa[poa.geometry.intersects(ring_geom_m)]
    # Join POA total persons onto the touching POAs and sum (v1's naive approach)
    merged = touching.merge(poa_pop_df[["POA_CODE21", "Total_P_P"]], on="POA_CODE21", how="left")
    return int(merged["Total_P_P"].sum())

def compare_v1_v2(v1_ring_pops: dict, v2_ring_pops: dict):
    """
    v1_ring_pops: {radius_m: naive_whole_postcode_pop} — v1's flawed approach
    v2_ring_pops: {radius_m: sa1_apportioned_pop} — v2's corrected approach
    Returns comparison DataFrame + renders grouped bar chart showing v1 overstatement.
    """
    import pandas as pd
    import matplotlib.pyplot as plt
    rows = []
    for r, v2_pop in v2_ring_pops.items():
        v1_pop = v1_ring_pops.get(r, 0)
        rows.append({"ring_km": r//1000, "v1_naive": v1_pop, "v2_apportioned": v2_pop,
                     "diff": v1_pop - v2_pop,
                     "pct_overstate": (v1_pop/v2_pop - 1)*100 if v2_pop else None})
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    # Grouped bar chart
    fig, ax = plt.subplots(figsize=(8, 4))
    x = range(len(df))
    ax.bar([i-0.2 for i in x], df["v1_naive"], width=0.4, label="v1 naive (whole-postcode)", color="#d62728")
    ax.bar([i+0.2 for i in x], df["v2_apportioned"], width=0.4, label="v2 apportioned (SA1)", color="#2ca02c")
    ax.set_xticks(list(x)); ax.set_xticklabels([f"{r} km" for r in df["ring_km"]])
    ax.set_ylabel("Catchment population"); ax.legend(); ax.set_title("v1 vs v2 catchment population")
    plt.show()
    return df
print("[compare] compare_v1_v2(v1_ring_pops, v2_ring_pops) defined — called in §3.3 after both v1 and v2 totals are computed")

## §2.5 Catchment Maps

**Folium** is interactive (inline only, NOT in the PDF — it's HTML/JS and
won't render in weasyprint, STACK.md constraint). The **static matplotlib +
contextily** twin handles the PDF.

One static figure per ring (1 km, 3 km, 5 km), each zoomed to its ring (D-27).
The Yarra River is annotated on every map so readers see the uniform-density
caveat's geographic issue (D-05).

In [ ]:
# Folium interactive map — inline only, NOT in PDF (D-25, D-30, STACK.md)
import folium
m_catch = folium.Map(location=[site_lat, site_lon], zoom_start=12, tiles="CartoDB Positron")

# Site pin
folium.Marker([site_lat, site_lon], popup=SITE_ADDRESS,
              icon=folium.Icon(color="red", icon="info-sign")).add_to(m_catch)

# 1/3/5 km buffer rings (reproject to EPSG:4326 for folium)
for r in BASE_ASSUMPTIONS["catchment_radii_m"]:
    buf_4326 = gpd.GeoDataFrame(geometry=buffers[r], crs="EPSG:7855").to_crs("EPSG:4326")
    folium.GeoJson(buf_4326, style_function=lambda f, r=r: {
        "color": {1000:"blue", 3000:"orange", 5000:"green"}[r],
        "weight": 2, "fillOpacity": 0.05
    }, name=f"{r//1000} km ring").add_to(m_catch)

# POA boundaries (peers) — shaded by apportionment share (computed in §3)
poa_peers_4326 = poa[poa["POA_CODE21"].isin(PEER_POSTCODES)].to_crs("EPSG:4326")
folium.GeoJson(poa_peers_4326, style_function=lambda f: {"color":"purple","weight":1,"fillOpacity":0.1},
               name="Peer POAs").add_to(m_catch)

# Yarra River line (D-05) — approximate path through Abbotsford
# NOTE: exact Yarra geometry comes from OSM or a local GeoJSON; for MVP, a hand-drawn
# polyline through known points is acceptable. Plan 02-02 may upgrade to OSM fetch.
yarra_pts = [[-37.808, 145.003], [-37.810, 145.000], [-37.815, 144.998], [-37.820, 144.995]]
folium.PolyLine(yarra_pts, color="aqua", weight=4, opacity=0.7, tooltip="Yarra River (approx)").add_to(m_catch)

folium.LayerControl().add_to(m_catch)
print("[map] Folium catchment map rendered (inline only — PDF uses static matplotlib below)")
m_catch

In [ ]:
# Static matplotlib + contextily figures for the PDF (D-27, D-29) — one per ring
import matplotlib.pyplot as plt
import contextily as cx
import numpy as np

site_4326 = site_point.to_crs("EPSG:4326")
poa_peers_4326 = poa[poa["POA_CODE21"].isin(PEER_POSTCODES)].to_crs("EPSG:4326")

for r in BASE_ASSUMPTIONS["catchment_radii_m"]:
    fig, ax = plt.subplots(figsize=(8, 8))
    # Buffer ring
    buf_4326 = gpd.GeoDataFrame(geometry=buffers[r], crs="EPSG:7855").to_crs("EPSG:4326")
    buf_4326.boundary.plot(ax=ax, color={1000:"blue", 3000:"orange", 5000:"green"}[r], linewidth=2)
    # POA boundaries
    poa_peers_4326.boundary.plot(ax=ax, color="purple", linewidth=0.8, alpha=0.6)
    # Site pin
    site_4326.plot(ax=ax, color="red", markersize=40)
    # Yarra River (approx)
    yarra_x = [145.003, 145.000, 144.998, 144.995]
    yarra_y = [-37.808, -37.810, -37.815, -37.820]
    ax.plot(yarra_x, yarra_y, color="aqua", linewidth=3, alpha=0.7, label="Yarra River (approx)")
    # Basemap
    cx.add_basemap(ax, crs="EPSG:4326", source=cx.providers.CartoDB.Positron)
    ax.set_title(f"Catchment — {r//1000} km ring (Johnston St site)")
    ax.legend(loc="upper right")
    plt.show()

## §2.6 Uniform-Density Caveat

> **Caveat (uniform-density assumption):** Catchment population is
> area-apportioned at SA1 level assuming population is uniformly distributed
> within each SA1. This likely *overstates* population in the Yarra River /
> industrial slice of Abbotsford (non-residential land). The Yarra River is
> annotated on every catchment map so readers can see the geographic issue.
> Mesh-block apportionment (finer) is the rigorous alternative — noted in
> PITFALLS.md Pitfall 2 as the "best" option; deferred as overkill for an
> MVP investor report.

# §3 Demographics

v1 had **no census staleness handling** (PITFALLS.md Pitfall 15) and
**conflated POA/SA2/SA3 geographies** (Pitfall 5). This section fetches
ABS G01/G02/G04 at POA level for peer benchmarking + SA1 level for
catchment apportionment, with a **local GCP fallback** that emits the SAME
tidy schema (D-17) so downstream code is source-agnostic.

**ERP scaling** (2021 → 2024) addresses the 4-5 year census lag — Abbotsford
has grown since the 2021 Census. The peer benchmarking table includes
**placeholder GP/pharmacy columns** ("— Phase 3") that Phase 3 fills —
no Places API calls in Phase 2.

Critically, §3 also **wires the SA1 total persons into the §2.3
apportionment function** — without this, the v2 catchment population totals
and the v1-vs-v2 comparison are stubs. This section completes the headline
v1-flaw-fix teaching moment.

In [ ]:
# ABS Data API — G01/G02 for POA 3067 + 9 peers (DEMO-01, D-13, D-14)
# All calls through the Phase 1 CachedSession (ARCHITECTURE.md Pattern 1)
import pandas as pd
import io

ABS_BASE = "https://data.api.abs.gov.au/rest"
PEER_POA_CODES = "+".join(PEER_POSTCODES)  # SDMX OR operator within a dimension

def fetch_abs_csv(flow_id, data_key):
    """Fetch ABS dataflow as CSV-with-labels through the cached session. Returns tidy DataFrame."""
    url = f"{ABS_BASE}/data/{flow_id}/{data_key}?format=csvfilewithlabels"
    resp = session.get(url)
    resp.raise_for_status()
    return pd.read_csv(io.StringIO(resp.text)), resp

def check_dataflow_exists(flow_id):
    """Check if a dataflow ID exists in the ABS dataflow list (D-11).
    Safe default: returns True on any failure so the fetch is always attempted —
    downstream functions have their own try/except fallbacks (no-hard-fail principle).
    The dataflow endpoint returns SDMX-ML XML, not JSON, so .json() would crash (CR-01 fix)."""
    try:
        flows_resp = session.get(f"{ABS_BASE}/dataflow?detail=allstubs")
        # Endpoint returns SDMX-ML XML (verified by §1.2 smoke test) — parse with ElementTree
        import xml.etree.ElementTree as ET
        root = ET.fromstring(flows_resp.text)
        # SDMX-ML namespace for dataflow structures
        ns = {"str": "http://www.sdmx.org/resources/sdmxml/schemas/v2_1/structure"}
        flow_ids = [f.get("id", "") for f in root.findall(".//str:Dataflow", ns)]
        if not flow_ids:
            # Namespace might differ — try without namespace
            flow_ids = [f.get("id", "") for f in root.iter() if "Dataflow" in f.tag]
        return flow_id in flow_ids
    except Exception as e:
        print(f"⚠ check_dataflow_exists({flow_id}) failed: {e} — defaulting to True (will attempt fetch)")
        return True

def parse_gcp_g01_to_tidy(xlsx_path):
    """Parse local GCP POA 3067 xlsx into the same schema as the API path (D-17).
    Schema parity: column names, dtypes, row-per-POA shape must match API path.
    Returns empty DataFrame with correct schema if xlsx parsing is not yet implemented —
    zero is not a valid population (WR-02 fix)."""
    print(f"⚠ parse_gcp_g01_to_tidy: xlsx cell parsing not yet implemented — returning empty schema")
    print(f"  GCP fallback values are not populated; peer table will show N/A for 3067 (WR-02)")
    df = pd.DataFrame(columns=["POA_CODE21", "Total_P_P", "M_P", "F_P", "Total_Dwll_D"])
    return df

def fetch_g01_poa():
    """G01: total persons, sex split, dwelling counts (D-13)."""
    try:
        # dataKey dimension order is runtime-verified via the datastructure endpoint
        # Pattern: {MEASURE}+{codes}.{POA}+{codes}.{FREQ}.{TIME}
        df, resp = fetch_abs_csv("ABS,C21_G01_POA,latest", f"all.{PEER_POA_CODES}.A.2021")
        print(f"[abs] G01 POA: {len(df)} rows from API (from_cache={resp.from_cache})")
        return df, "api"
    except Exception as e:
        print(f"⚠ ABS G01 POA API failed: {e}")
        print(f"  Falling back to local GCP_POA3067.xlsx (D-15, D-16)")
        print(f"  Peer comparison degraded — 9 peer rows marked N/A")
        # Local fallback — parse GCP xlsx into the SAME tidy schema (D-17)
        xlsx_path = PROJECT_ROOT / "data" / "local" / "GCP_POA3067.xlsx"
        if xlsx_path.exists():
            df = parse_gcp_g01_to_tidy(xlsx_path)
            return df, "fallback"
        else:
            print(f"⚠ GCP fallback file also missing: {xlsx_path}")
            return pd.DataFrame(), "none"

def fetch_g02_poa():
    """G02: median age, median income (D-14)."""
    try:
        df, resp = fetch_abs_csv("ABS,C21_G02_POA,latest", f"all.{PEER_POA_CODES}.A.2021")
        print(f"[abs] G02 POA: {len(df)} rows from API (from_cache={resp.from_cache})")
        return df, "api"
    except Exception as e:
        print(f"⚠ ABS G02 POA API failed: {e}")
        print(f"  Falling back to local GCP_POA3067.xlsx (D-15, D-16)")
        print(f"  Peer comparison degraded — 9 peer rows marked N/A")
        xlsx_path = PROJECT_ROOT / "data" / "local" / "GCP_POA3067.xlsx"
        if xlsx_path.exists():
            # Parse G02 medians from GCP xlsx — schema parity with API path (D-17, WR-02)
            print(f"⚠ G02 GCP fallback: xlsx cell parsing not yet implemented — returning empty schema (WR-02)")
            df = pd.DataFrame(columns=["POA_CODE21", "Median_age_persons", "Median_tot_hshld_inc_weekly"])
            return df, "fallback"
        else:
            print(f"⚠ GCP fallback file also missing: {xlsx_path}")
            return pd.DataFrame(), "none"

g01_df, g01_source = fetch_g01_poa()
print(f"[abs] G01 source: {g01_source}, shape: {g01_df.shape}")

g02_df, g02_source = fetch_g02_poa()
print(f"[abs] G02 source: {g02_source}, shape: {g02_df.shape}")

In [ ]:
# G04 age by sex — runtime-verify C21_G04_POA existence (D-11, D-12)
# v1 had no age-band data — this feeds the Phase 3 demand model (age-band attendance rates)
# Age bands use ABS standard 5-year structure: 0-4, 5-9, ..., 80-84, 85+ (D-18)
G04_POA_EXISTS = check_dataflow_exists("C21_G04_POA")
print(f"[abs] C21_G04_POA exists: {G04_POA_EXISTS}")

def fetch_g04_poa():
    """G04: age by sex in ABS standard 5-year bands (D-18)."""
    try:
        df, resp = fetch_abs_csv("ABS,C21_G04_POA,latest", f"all.{PEER_POA_CODES}.A.2021")
        print(f"[abs] G04 POA: {len(df)} rows from API (from_cache={resp.from_cache})")
        return df, "api"
    except Exception as e:
        print(f"⚠ ABS G04 POA API failed: {e}")
        print(f"  Falling back to G04 fallback (D-12) — consistent with G01/G02 fallback (WR-03 fix)")
        return fetch_g04_fallback()

def fetch_g04_fallback():
    """G04 fallback if C21_G04_POA not available at POA level (D-12).
    Fallback: C21_G04_SA2 + apportion, or local GCP for 3067 65+ share with peers N/A."""
    print("⚠ C21_G04_POA not available — using fallback (D-12)")
    print("  Fallback: local GCP for 3067 65+ share, peers marked N/A")
    # Return a stub DataFrame with pct_65plus for 3067 only
    df = pd.DataFrame([{
        "POA_CODE21": "3067",
        "pct_65plus": 0,  # populated from local GCP if available
    }])
    return df, "fallback"

if G04_POA_EXISTS:
    g04_df, g04_source = fetch_g04_poa()
    # Derive 65+ share: sum bands >= 65 / total (D-18 — ABS standard 5-year bands)
    # Age bands: 0-4, 5-9, 10-14, ..., 60-64, 65-69, 70-74, 75-79, 80-84, 85+
    # Use explicit prefix matching, not fragile substring matching (WR-04 fix)
    age_bands_65plus = [b for b in g04_df.columns
                        if b.startswith("Age_yr_65") or b.startswith("Age_yr_70")
                        or b.startswith("Age_yr_75") or b.startswith("Age_yr_80")
                        or b.startswith("Age_yr_85") or "Age_yr_85ov" in b
                        or b in ("Age_65_69", "Age_70_74", "Age_75_79", "Age_80_84", "Age_85plus")]
    print(f"[abs] G04 65+ age bands matched: {age_bands_65plus}")
    if not age_bands_65plus:
        # Fallback: print all columns for debugging if no bands matched (teaching transparency)
        print(f"⚠ No 65+ age bands matched — G04 columns: {list(g04_df.columns)}")
        print(f"  Check ABS G04 data dictionary for exact column names (WR-04)")
    if age_bands_65plus and "Total_P_P" in g04_df.columns:
        g04_df["pct_65plus"] = g04_df[age_bands_65plus].sum(axis=1) / g04_df["Total_P_P"] * 100
    else:
        print("⚠ Could not derive pct_65plus from G04 columns — check schema")
        g04_df["pct_65plus"] = 0
else:
    g04_df, g04_source = fetch_g04_fallback()

print(f"[abs] G04 source: {g04_source}, shape: {g04_df.shape}")

In [ ]:
# SA1 total persons — for the §2.3 apportionment weights (D-03, D-08)
# Spatial-filter SA1s intersecting 5km buffer first (beat 30s timeout — PITFALLS.md Pitfall 4)
# This wires the SA1 population into the apportionment function to compute
# actual v2 catchment population totals — the headline v1-flaw fix completion.
if USE_SA1:
    sa1_codes = sa1_in_5k["SA1_CODE21"].tolist()
    print(f"[abs] Fetching SA1 total persons for {len(sa1_codes)} SA1s")

    SA1_G01_EXISTS = check_dataflow_exists("C21_G01_SA1")
    print(f"[abs] C21_G01_SA1 exists: {SA1_G01_EXISTS}")

    if SA1_G01_EXISTS:
        # Fetch C21_G01_SA1 for only the spatial-filtered SA1 IDs (D-08)
        sa1_codes_or = "+".join(sa1_codes)
        try:
            sa1_pop_df, sa1_resp = fetch_abs_csv("ABS,C21_G01_SA1,latest", f"all.{sa1_codes_or}.A.2021")
            sa1_pop_df = sa1_pop_df[["SA1_CODE21", "Total_P_P"]]  # tidy schema
            sa1_source = "api"
            print(f"[abs] SA1 total persons: {len(sa1_pop_df)} rows, source={sa1_source} (from_cache={sa1_resp.from_cache})")
        except Exception as e:
            print(f"⚠ ABS C21_G01_SA1 API failed: {e}")
            print(f"  Falling back to SA1 GCP DataPack (D-03)")
            sa1_datapack = PROJECT_ROOT / "data" / "local" / "2021Census_G01_AUST_SA1.csv"
            if sa1_datapack.exists():
                sa1_pop_df = pd.read_csv(sa1_datapack)
                sa1_pop_df = sa1_pop_df[sa1_pop_df["SA1_CODE21"].isin(sa1_codes)][["SA1_CODE21", "Total_P_P"]]
                sa1_source = "datapack"
                print(f"[abs] SA1 total persons: {len(sa1_pop_df)} rows, source={sa1_source}")
            else:
                print(f"⚠ SA1 DataPack missing: {sa1_datapack}")
                print(f"  Download from ABS Census DataPacks (SA1 GCP) — see .env.example")
                sa1_pop_df = None
                sa1_source = "none"
    else:
        print("⚠ C21_G01_SA1 not available — falling back to SA1 GCP DataPack (D-03)")
        sa1_datapack = PROJECT_ROOT / "data" / "local" / "2021Census_G01_AUST_SA1.csv"
        if sa1_datapack.exists():
            sa1_pop_df = pd.read_csv(sa1_datapack)
            sa1_pop_df = sa1_pop_df[sa1_pop_df["SA1_CODE21"].isin(sa1_codes)][["SA1_CODE21", "Total_P_P"]]
            sa1_source = "datapack"
            print(f"[abs] SA1 total persons: {len(sa1_pop_df)} rows, source={sa1_source}")
        else:
            print(f"⚠ SA1 DataPack missing: {sa1_datapack}")
            print(f"  Download from ABS Census DataPacks (SA1 GCP) — see .env.example")
            sa1_pop_df = None
            sa1_source = "none"

    if sa1_pop_df is not None:
        # Wire into §2.3 apportionment — compute actual v2 catchment population totals
        v2_ring_pops = {}
        v1_ring_pops = {}
        for r in BASE_ASSUMPTIONS["catchment_radii_m"]:
            pop, inter = apportion_ring(sa1_in_5k, sa1_pop_df, buffers[r].iloc[0])
            v2_ring_pops[r] = pop
            print(f"[catchment] {r//1000} km ring: v2 apportioned pop = {pop:,.0f}")

            # v1 naive: sum FULL POA population for any POA touching the buffer (CR-02 fix)
            # Uses g01_df from §3.1 — the POA total persons already fetched
            if g01_df is not None and len(g01_df) > 0 and "POA_CODE21" in g01_df.columns:
                v1_pop = v1_naive_catchment_pop(buffers[r].iloc[0], g01_df)
                v1_ring_pops[r] = v1_pop
                print(f"[catchment] {r//1000} km ring: v1 naive pop = {v1_pop:,.0f} (whole-postcode sum)")
            else:
                print(f"⚠ g01_df empty — v1 naive pop unavailable for {r//1000} km ring")
                v1_ring_pops[r] = 0

        # Plausibility assertion (D-09b, PITFALLS.md Pitfall 2)
        lo, hi = BASE_ASSUMPTIONS["catchment_pop_plausible_range"]
        assert lo < v2_ring_pops[3000] < hi, \
            f"3km catchment pop {v2_ring_pops[3000]:,.0f} outside plausible range ({lo:,}-{hi:,}) — PITFALLS.md Pitfall 2"
        print(f"[catchment] ✓ 3km pop plausibility assertion passed ({v2_ring_pops[3000]:,.0f})")

        # Call the §2.4 v1-vs-v2 comparison with BOTH v1 and v2 totals (D-06 completion, CR-02 fix)
        if v1_ring_pops:
            comparison_df = compare_v1_v2(v1_ring_pops, v2_ring_pops)
        else:
            print("[catchment] ⚠ v1 naive pops unavailable — v1-vs-v2 comparison skipped")
    else:
        print("[catchment] ⚠ SA1 pop unavailable — v2 totals + v1 comparison deferred")
        v2_ring_pops = {}
        v1_ring_pops = {}
else:
    print("[catchment] ⚠ SA1 shapefile missing — v2 totals deferred (POA-level fallback)")
    v2_ring_pops = {}
    v1_ring_pops = {}

## §3.4 ERP Scaling (Census Staleness)

The 2021 Census is ~4-5 years old (PITFALLS.md Pitfall 15). Abbotsford has
grown. **ERP (Estimated Resident Population)** is the ABS's official annual
update. We scale 2021 census → 2024 ERP-adjusted using the Yarra SA2 growth
rate (D-21 — uniform rate is defensible since the catchment is largely within
Yarra SA2).

The caveat is printed and both raw + scaled values are shown side-by-side
(D-22). ERP is fetched through the same CachedSession (D-23).

ERP scaling is applied to BOTH the catchment population AND the peer
benchmarking table (D-24) — consistent treatment.

In [ ]:
# ERP scaling — 2021 census → 2024 ERP-adjusted (DEMO-04, D-19..D-24, PITFALLS.md Pitfall 15)
# Source: ABS_ANNUAL_ERP_ASGS2021 (annual ERP by state and sub-state geography)
ERP_VINTAGE = BASE_ASSUMPTIONS["erp_vintage_year"]  # 2024 (2023-24 FY, released March 2025)

def fetch_erp_sa2():
    """Fetch ERP by SA2 from ABS_ANNUAL_ERP_ASGS2021. Returns DataFrame with SA2 code, year, population."""
    try:
        # dataKey: {MEASURE}.{REGION}+{SA2_codes}.{FREQ}.{TIME}
        # Yarra SA2 code — verify at runtime via datastructure introspection
        # ASGS Edition 3 Yarra SA2 code: 206041001 (verify at runtime)
        df, resp = fetch_abs_csv("ABS,ABS_ANNUAL_ERP_ASGS2021,latest", f"all.206041001.A.2021+2024")
        print(f"[erp] SA2 ERP fetched: {len(df)} rows (from_cache={resp.from_cache})")
        return df, "api"
    except Exception as e:
        print(f"⚠ ERP API failed: {e} — ERP scaling skipped (DEMO-04 degraded)")
        return None, "none"

erp_df, erp_source = fetch_erp_sa2()
erp_growth_rate = 1.0  # default: no scaling if ERP unavailable

if erp_df is not None:
    # Compute Yarra SA2 growth rate 2021 → 2024
    pop_2021 = erp_df[erp_df["TIME_PERIOD"]==2021]["OBS_VALUE"].iloc[0]
    pop_2024 = erp_df[erp_df["TIME_PERIOD"]==2024]["OBS_VALUE"].iloc[0]
    erp_growth_rate = pop_2024 / pop_2021
    print(f"[erp] Yarra SA2: 2021={pop_2021:,.0f} → 2024={pop_2024:,.0f} (rate={erp_growth_rate:.4f})")

    # Apply to catchment population (D-24)
    if v2_ring_pops:
        v2_ring_pops_erp = {r: pop * erp_growth_rate for r, pop in v2_ring_pops.items()}
        print(f"[erp] Catchment 3km: 2021={v2_ring_pops[3000]:,.0f} → 2024 ERP={v2_ring_pops_erp[3000]:,.0f}")
    else:
        v2_ring_pops_erp = {}
        print("[erp] Catchment pop not available — ERP scaling on catchment deferred")

    # Caveat (D-22) — printed markdown note
    print(f"\n[erp] ⚠ Caveat: Population scaled from 2021 Census baseline to {ERP_VINTAGE} ERP-adjusted")
    print(f"  Method: ABS Regional Population (3218.0) Yarra SA2 growth rate applied (D-21)")
    print(f"  Round aggressively — false precision is misleading (PITFALLS.md Pitfall 15)")
else:
    v2_ring_pops_erp = {}
    print("[erp] ⚠ ERP unavailable — peer table uses unscaled 2021 census values")

## §3.5 Peer Benchmarking Table

The peer set frames the market context (DEMO-02, DEMO-03). Census columns
now: population (raw 2021 + ERP-scaled), median age, 65+ share, median
income. GP count + pharmacy count are **placeholders** ("— Phase 3") for
Phase 3 to fill (D-26). No Places API calls in Phase 2.

The 10 peer postcodes are already in `BASE_ASSUMPTIONS["peer_postcodes"]`
from Phase 1. This table merges G01/G02/G04 into a single tidy table keyed
by POA.

In [ ]:
# Peer benchmarking table (DEMO-02, DEMO-03, D-26)
# Census columns now; GP/pharmacy columns are placeholders for Phase 3
peer_table = g01_df[["POA_CODE21", "Total_P_P"]].merge(
    g02_df[["POA_CODE21", "Median_age_persons", "Median_tot_hshld_inc_weekly"]],
    on="POA_CODE21", how="outer"
).merge(
    g04_df[["POA_CODE21", "pct_65plus"]],
    on="POA_CODE21", how="outer"
)

# ERP-scaled population column (D-24)
if erp_df is not None:
    peer_table["Total_P_P_erp"] = (peer_table["Total_P_P"] * erp_growth_rate).round(0)
else:
    peer_table["Total_P_P_erp"] = peer_table["Total_P_P"]  # unscaled fallback

# Placeholder columns for Phase 3 (D-26)
peer_table["gp_count"] = "— Phase 3"
peer_table["pharmacy_count"] = "— Phase 3"

# Highlight 3067 (site POA)
peer_table["is_site"] = peer_table["POA_CODE21"] == "3067"

print(peer_table.to_string(index=False))

## §3.6 Peer Comparison Charts

2×2 bar chart grid (D-29): population (ERP-scaled), median age, 65+ share,
median income. POA 3067 highlighted in each chart. This is the visual
peer-context frame for the investor report.

In [ ]:
# Peer comparison 2×2 bar chart grid (D-29)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
charts = [
    ("Total_P_P_erp", "Population (ERP-scaled)"),
    ("Median_age_persons", "Median age"),
    ("pct_65plus", "65+ share (%)"),
    ("Median_tot_hshld_inc_weekly", "Median weekly household income ($)"),
]
for ax, (col, title) in zip(axes.flat, charts):
    colors = ["#d62728" if s else "#2ca02c" for s in peer_table["is_site"]]
    ax.bar(peer_table["POA_CODE21"], peer_table[col], color=colors)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=45)
fig.suptitle("Peer postcode comparison — POA 3067 (site) highlighted", y=1.02)
plt.tight_layout()
plt.show()

## Next Steps

**§4 Competitors (Phase 3)** will fetch Google Places (New) data through the
cached session, fill the GP count + pharmacy count columns in the peer
table, and build the demand model using the catchment age profile from §3.

The v1-vs-v2 comparison (§2.4) is now complete with real v2 totals — the
headline teaching moment showing v1's whole-postcode summing inflated
catchment population 2–4×.